In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
path = '/Users/liufanling/Library/CloudStorage/OneDrive-Personal/1 UC DAVIS/2024 Summer/CSRA/Outputs/SYNGAP_T1/Network/'
file_csv = path + 'Compiled_Networks.csv'
data_df = pd.read_csv(file_csv)
df = data_df.replace(np.nan,0.0)
df.head()

In [ ]:
df.columns

In [ ]:
# check div
df['DIV'].unique()

In [ ]:
# subset the data to div 21
df_div22 = df[df['DIV']==22]
df_div22.head()

In [ ]:
# check NeuronType
neuron_types = df['NeuronType'].unique()
neuron_types

In [ ]:
# Bursting columns
num_columns = ['mean_IBI',
       'cov_IBI', 'mean_Burst_Peak', 'cov_Burst_Peak', 'Number_Bursts',
       'mean_Spike_per_Burst', 'cov_Spike_per_Burst', 'mean_Burst_Peak_Abs',
       'cov_Burst_Peak_Abs', 'mean_BurstDuration', 'cov_BurstDuration',
       'MeanNetworkISI', 'CoVNetworkISI', 'MeanWithinBurstISI',
       'CoVWithinBurstISI', 'MeanOutsideBurstISI', 'CoVOutsideBurstISI',
       'Fanofactor']
mean_column = ['mean_IBI',
       'mean_Burst_Peak', 'Number_Bursts',
       'mean_Spike_per_Burst', 'mean_Burst_Peak_Abs',
       'mean_BurstDuration',
       'MeanNetworkISI', 'MeanWithinBurstISI',
       'MeanOutsideBurstISI', 
       'Fanofactor']
cov_burst_column = ['cov_IBI','cov_Burst_Peak', 'cov_Spike_per_Burst', 'cov_Burst_Peak_Abs', 'CoVNetworkISI', 'CoVWithinBurstISI', 'CoVOutsideBurstISI', 'cov_BurstDuration']

In [ ]:
# Selected features to analysis
features = ['mean_Burst_Peak', 'Number_Bursts', 'mean_BurstDuration']

Questions: can we select more features? (Need to check correlation between features and avoid choose those featrues have high correlation.)

In [ ]:
# distribution of features
# color profile
color_map = {}
for n_type in neuron_types:
    if 'WT' in n_type:
        color_map[n_type] = 'blue'
    elif len(color_map) == 1:  # This assumes WT is always there and is the first one to get 'blue'
        color_map[n_type] = 'red'
    else:
        color_map[n_type] = 'grey'

# Plot histograms for each feature across genotypes
for neuron_type in neuron_types:
    # Filter data for the current neuron type
    type_df = df[df['NeuronType'] == neuron_type]

    # Plot histograms for each feature
    fig, axes = plt.subplots(nrows=1, ncols=len(features), figsize=(15, 5))
    for i, feature in enumerate(features):
        for n_type in neuron_types:
            subtype_df = df[df['NeuronType'] == n_type]
            axes[i].hist(subtype_df[feature], bins=10, alpha=0.5, label=n_type, color=color_map[n_type])
        axes[i].set_title(f'Distribution of {feature}')
        axes[i].set_xlabel(feature)
        axes[i].set_ylabel('Frequency')
        axes[i].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Plot correlation matrices for each neuron type
for neuron_type in neuron_types:
    # Filter data for the current neuron type
    type_df = df[df['NeuronType'] == neuron_type]
    
    # Calculate correlation matrix
    corr = type_df[features].corr()
    
    # Plotting the correlation matrix
    fig, ax = plt.subplots(figsize=(8, 5))
    cax = ax.matshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
    plt.xticks(range(len(features)), features, rotation=90)
    plt.yticks(range(len(features)), features)
    plt.colorbar(cax)
    plt.title(f'Correlation Matrix for {neuron_type}')

    # Adding correlation numbers to the plot
    for (i, j), val in np.ndenumerate(corr):
        # Choose text color for readability
        text_color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=text_color)

    plt.show()

In [ ]:
# Plot correlation matrices for each neuron type

for neuron_type in neuron_types:
    # Filter data for the current neuron type
    type_df = df_div22[df_div22['NeuronType'] == neuron_type]
    
    # Calculate correlation matrix
    corr = type_df[features].corr()
    
    # Plotting the correlation matrix
    fig, ax = plt.subplots(figsize=(8, 5))
    cax = ax.matshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
    plt.xticks(range(len(features)), features, rotation=90)
    plt.yticks(range(len(features)), features)
    plt.colorbar(cax)
    plt.title(f'Correlation Matrix for {neuron_type}')

    # Adding correlation numbers to the plot
    for (i, j), val in np.ndenumerate(corr):
        # Choose text color for readability
        text_color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=text_color)

    plt.show()

Choose a specific div, the correlation increase, should be careful about this. Also, can we apply time series to this problem? 

**PCA (need improvement)**

In [ ]:
# PCA analysis on burst features
# Standardizing the data
scaler = StandardScaler()
scaler.fit(df[features])
data_scaled = scaler.transform(df[features])

# Performing PCA
pca = PCA()
pca_result = pca.fit_transform(data_scaled)

# Explained variance ratio
explained_variance = pca.explained_variance_ratio_

# Plotting the explained variance ratio
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(explained_variance) + 1), explained_variance, marker='o', linestyle='--')
plt.title('Explained Variance by Principal Components')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.grid(True)
plt.show()

In [ ]:
# Cumulative explained variance
cumulative_explained_variance = explained_variance.cumsum()

# Plotting the cumulative explained variance
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_explained_variance) + 1), cumulative_explained_variance, marker='o', linestyle='--')
plt.title('Cumulative Explained Variance by Principal Components')
plt.xlabel('Principal Component')
plt.ylabel('Cumulative Explained Variance')
plt.grid(True)
plt.show()

In [ ]:
# Principal component scores (i.e., transformed data)
pca_scores = pd.DataFrame(pca_result, columns=[f'PC{i+1}' for i in range(len(explained_variance))])

# Principal component loadings (i.e., how much each feature contributes to each PC)
pca_loadings = pd.DataFrame(pca.components_.T, columns=[f'PC{i+1}' for i in range(len(explained_variance))], index=features)

# Display the first few rows of the PCA scores and loadings
pca_scores.head(), pca_loadings

In [ ]:
# Creating a biplot
def biplot(scores, loadings, feature_labels, pc1=0, pc2=1):
    plt.figure(figsize=(10, 7))
    
    # Plotting the sample scores
    plt.scatter(scores[:, pc1], scores[:, pc2], alpha=0.5, label='Samples')
    
    # Plotting the feature vectors (loadings)
    for i, feature in enumerate(feature_labels):
        # Ensure the loadings are correctly accessed and plotted
        plt.arrow(0, 0, loadings[i, pc1], loadings[i, pc2], color='r', alpha=0.7, head_width=0.02)
        plt.text(loadings[i, pc1] * 1.2, loadings[i, pc2] * 1.2, feature, color='blue', ha='center', va='center')
    
    plt.xlabel(f'Principal Component {pc1 + 1}')
    plt.ylabel(f'Principal Component {pc2 + 1}')
    plt.title('PCA Biplot')
    plt.grid(True)
    plt.legend(['Samples', 'Features'], loc='best')
    plt.show()

# Correctly plotting the biplot for the first two principal components with feature labels
biplot(pca_result, pca.components_,features)

**Logistic Regression**

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Load and preprocess data
file_csv = '/Users/liufanling/Library/CloudStorage/OneDrive-Personal/1 UC DAVIS/2024 Summer/CSRA/Outputs/SYNGAP_ALL/Network/Compiled_Networks.csv'
data = pd.read_csv(file_csv)
data = data.replace(np.nan, 0.0)

# Features and target
X = data[['mean_Burst_Peak', 'Number_Bursts', 'mean_BurstDuration']]
y = data['NeuronType']

# Convert categorical variables to dummy/indicator variables
y_dummy = pd.get_dummies(y, drop_first=False)

# Standardizing the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_dummy, test_size=0.2, random_state=42)

# Fit the model using statsmodels
X_train_sm = sm.add_constant(X_train)  # Adding a constant for the intercept
model = sm.MNLogit(y_train, X_train_sm)
result = model.fit()

# Display summary
print(result.summary())

Three features has significance contribute to the logistic model

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# Load and preprocess data
file_csv = '/Users/liufanling/Library/CloudStorage/OneDrive-Personal/1 UC DAVIS/2024 Summer/CSRA/Outputs/SYNGAP_ALL/Network/Compiled_Networks.csv'
data = pd.read_csv(file_csv)
data = data.replace(np.nan, 0.0)

# Features and target
X = data[['mean_Burst_Peak', 'Number_Bursts', 'mean_BurstDuration']]
y = data['NeuronType']

# Standardizing the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Initialize and train the logistic regression model
model = LogisticRegression(multi_class='multinomial', solver='lbfgs')
model.fit(X_train, y_train)

# Predicting and evaluating the model
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

**LDA**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Load and preprocess data
file_csv = '/Users/liufanling/Library/CloudStorage/OneDrive-Personal/1 UC DAVIS/2024 Summer/CSRA/Outputs/SYNGAP_ALL/Network/Compiled_Networks.csv'
data = pd.read_csv(file_csv)
data = data.replace(np.nan, 0.0)

# Features and target
X = data[['mean_Burst_Peak', 'Number_Bursts', 'mean_BurstDuration']]
y = data['NeuronType']

# Standardizing the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Initialize and train the LDA model
lda = LDA()
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

# Predicting and evaluating the model
y_pred = lda.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

# Load the dataset
file_csv = '/Users/liufanling/Library/CloudStorage/OneDrive-Personal/1 UC DAVIS/2024 Summer/CSRA/Outputs/SYNGAP_ALL/Network/Compiled_Networks.csv'
data = pd.read_csv(file_csv)
data = data.replace(np.nan, 0.0)

# Specify features to be included in the LDA
features = ['mean_IBI', 'cov_IBI', 'mean_Burst_Peak', 'cov_Burst_Peak',
            'mean_Burst_Peak_Abs', 'cov_Burst_Peak_Abs', 'Number_Bursts',
            'mean_Spike_per_Burst', 'cov_Spike_per_Burst', 'mean_BurstDuration',
            'MeanNetworkISI', 'CoVNetworkISI', 'MeanWithinBurstISI', 'CoVWithinBurstISI', 
            'MeanOutsideBurstISI', 'CoVOutsideBurstISI', 'Fanofactor']
X = data[features]  # Ensure only desired, numeric features are included

# Encode the target variable
le = LabelEncoder()
y = le.fit_transform(data['NeuronType'])
print(f"Unique classes in 'NeuronType': {len(set(y))}")  # Check number of unique classes

# Apply LDA
lda = LDA()
X_lda = lda.fit_transform(X, y)
print(f"Number of Linear Discriminants: {X_lda.shape[1]}")  # Check how many LDs were produced

# Create a DataFrame for LDA results
if X_lda.shape[1] > 1:
    lda_df = pd.DataFrame(X_lda, columns=['LD1', 'LD2'])
else:
    lda_df = pd.DataFrame(X_lda, columns=['LD1'])
lda_df['NeuronType'] = le.inverse_transform(y)  # Convert labels back to original for plotting

In [ ]:
# Density plot of LDA results (1 dimension)
plt.figure(figsize=(10, 6))
sns.kdeplot(x='LD1', hue='NeuronType', data=lda_df, palette='Set1', common_norm=False)
plt.xlabel('Linear Discriminant 1')
plt.ylabel('Density')
plt.title('Density Plot of LDA Results')
plt.legend(loc='best')
plt.show()

In [ ]:
# Density plot of LDA results (1 dimension)

import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

# Load the dataset
file_csv = '/Users/liufanling/Library/CloudStorage/OneDrive-Personal/1 UC DAVIS/2024 Summer/CSRA/Outputs/SYNGAP_ALL/Network/Compiled_Networks.csv'
data = pd.read_csv(file_csv)
data = data.replace(np.nan, 0.0)

# Specify features to be included in the LDA
# features = ['mean_IBI', 'cov_IBI', 'mean_Burst_Peak', 'cov_Burst_Peak',
#             'mean_Burst_Peak_Abs', 'cov_Burst_Peak_Abs', 'Number_Bursts',
#             'mean_Spike_per_Burst', 'cov_Spike_per_Burst', 'mean_BurstDuration',
#             'MeanNetworkISI', 'CoVNetworkISI', 'MeanWithinBurstISI', 'CoVWithinBurstISI', 
#             'MeanOutsideBurstISI', 'CoVOutsideBurstISI', 'Fanofactor']

features = ['mean_IBI', 'mean_Burst_Peak', 
            'mean_Burst_Peak_Abs',  'Number_Bursts',
            'mean_Spike_per_Burst', 'mean_BurstDuration',
            'MeanNetworkISI', 'MeanWithinBurstISI', 
            'MeanOutsideBurstISI','Fanofactor']

X = data[features]

# Encode the target variable
le = LabelEncoder()
y = le.fit_transform(data['NeuronType'])
y_labels = le.inverse_transform(y)
print(f"Unique classes in 'NeuronType': {len(set(y))}")  # Check number of unique classes

# Apply LDA
lda = LDA()
X_lda = lda.fit_transform(X, y)
print(f"Number of Linear Discriminants: {X_lda.shape[1]}")  # Check how many LDs were produced

# Create a DataFrame for LDA results
lda_df = pd.DataFrame(X_lda, columns=['LD1'])
lda_df['NeuronType'] = y_labels

# Plot the LDA results
fig, ax = plt.subplots(figsize=(10, 6))
palette = sns.color_palette("Set1", len(np.unique(y_labels)))
sns.scatterplot(x='LD1', y=np.zeros(len(lda_df)), hue='NeuronType', data=lda_df, palette=palette, s=50, edgecolor='k', legend=False)

# Kernel density estimate and plot
medians = {}
for label, color in zip(np.unique(y_labels), palette):
    class_data = lda_df[lda_df['NeuronType'] == label]['LD1']
    kde = stats.gaussian_kde(class_data)
    x_range = np.linspace(lda_df['LD1'].min(), lda_df['LD1'].max(), 100)
    y_values = kde(x_range)
    y_normalized = y_values / y_values.max() * 0.4  # Normalize the peak for visualization
    ax.fill_between(x_range, y_normalized, alpha=0.3, color=color, label=f'{label} Density')
    # Calculate and store medians
    medians[label] = np.median(class_data)

# Plot vertical lines for medians and add text annotations
for label, color in zip(np.unique(y_labels), palette):
    median = medians[label]
    ax.axvline(median, color=color, linestyle='--', linewidth=1)
    ax.text(median, 0.45, f'{label}\nMedian: {median:.2f}', color=color, ha='center', va='top', fontsize=8, rotation=90)

# Calculate distances between medians and plot these
sorted_medians = sorted(medians.items(), key=lambda x: x[1])
for i in range(len(sorted_medians) - 1):
    label1, median1 = sorted_medians[i]
    label2, median2 = sorted_medians[i + 1]
    distance = median2 - median1
    ax.text((median1 + median2) / 2, 0.1, f'Dist: {distance:.2f}', ha='center', va='bottom', fontsize=8, color='black')
    ax.hlines(0.1, median1, median2, colors='gray', linestyles='--', linewidth=1)

# Custom legend creation
handles, labels = ax.get_legend_handles_labels()
custom_legend = [plt.Line2D([0], [0], marker='o', color='w', label=label, markersize=8, markerfacecolor=color, linestyle='None') for label, color in zip(np.unique(y_labels), palette)]
density_legend = [plt.Line2D([0], [0], color=color, label=f'{label} Density', linewidth=10, alpha=0.3) for label, color in zip(np.unique(y_labels), palette)]
plt.legend(handles=custom_legend, loc='upper right', title='Groups')

ax.set_title('LDA Analysis of Neuron Types Using Medians')
ax.set_xlabel('Linear Discriminant 1')
ax.set_ylabel('Density')
ax.set_ylim([-0.05, 0.5])
plt.show()

If there are more than three genotypes, we can only plot LD1 and LD2
Try multiclass by using biplot, Projections, Partition Plot, or dimensionality reduction techniques for visualization (like PCA, t-SNE, or UMAP) after performing LDA.

In [ ]:
# Use Iris dataset as example
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import seaborn as sns

# Load the Iris dataset
data = sns.load_dataset('iris')

# Preprocess the data
X = data.drop('species', axis=1)
y = data['species']
feature_names = X.columns  # Dynamically get feature names

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply LDA
lda = LDA(n_components=2)  # Limit to 2 components for visualization
X_lda = lda.fit_transform(X_scaled, y)

# Create a DataFrame for the transformed data
lda_df = pd.DataFrame(X_lda, columns=['LD1', 'LD2'])
lda_df['species'] = y

# Calculate scaling factor for 95% confidence ellipse (Chi-square value)
scaling_factor = np.sqrt(5.991)

# Retrieve the explained variance ratio from the LDA model
explained_variance_ratio = lda.explained_variance_ratio_ * 100  # Convert to percentage

# Plotting
plt.figure(figsize=(10, 7))

# Define a specific color palette from seaborn for clarity
palette = sns.color_palette("Set1", n_colors=len(lda_df['species'].unique()))

# Map species to the palette
color_map = dict(zip(lda_df['species'].unique(), palette))

# Plot the LDA data with corrected species colors
sns.scatterplot(x='LD1', y='LD2', hue='species', data=lda_df, palette=color_map, s=100, alpha=0.7)

# Add ellipses with corrected colors
for species in lda_df['species'].unique():
    sub_data = lda_df[lda_df['species'] == species]
    covariance = np.cov(sub_data[['LD1', 'LD2']].T)
    lambda_, v = np.linalg.eig(covariance)
    lambda_ = np.sqrt(lambda_) * scaling_factor  # Adjust the eigenvalues
    ell = Ellipse(xy=(np.mean(sub_data['LD1']), np.mean(sub_data['LD2'])),
                  width=lambda_[0]*2, height=lambda_[1]*2,
                  angle=np.rad2deg(np.arccos(v[0, 0])), edgecolor=color_map[species], fc='None', lw=2, linestyle='--')
    plt.gca().add_patch(ell)

# Adjusting vector scaling for better fit
scale_factor = 2  # Same scale factor for consistency
for i, feature in enumerate(feature_names):
    plt.arrow(0, 0, lda.scalings_[i, 0]*scale_factor, lda.scalings_[i, 1]*scale_factor, color='black', head_width=0.05, head_length=0.1)
    plt.text(lda.scalings_[i, 0]*scale_factor*1.2, lda.scalings_[i, 1]*scale_factor*1.2, feature, color='black')

plt.xlabel(f'LD1 ({explained_variance_ratio[0]:.2f}%)')
plt.ylabel(f'LD2 ({explained_variance_ratio[1]:.2f}%)')
plt.title('LDA of Iris Dataset')
plt.legend(title='Species')
plt.grid(True)
plt.show()

In [ ]:
# Generate a synthetic neuron dataset for LDA visualization
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import seaborn as sns

# Generate a synthetic dataset
X, y = make_classification(n_samples=150, n_features=4, n_informative=4, n_redundant=0, n_clusters_per_class=1, n_classes=3, random_state=42)
class_labels = ['WT', 'HET', 'HOM']  # Classes
y_labels = [class_labels[label] for label in y]  # Convert numerical labels to string labels using the class_labels mapping

# Create a DataFrame for the features and target
feature_names = ['Feature1', 'Feature2', 'Feature3', 'Feature4']
data = pd.DataFrame(X, columns=feature_names)
data['NeuronType'] = y_labels

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data[feature_names])

# Apply LDA
lda = LDA(n_components=2)  # Limit to 2 components for visualization
X_lda = lda.fit_transform(X_scaled, data['NeuronType'])

# Create a DataFrame for the transformed data
lda_df = pd.DataFrame(X_lda, columns=['LD1', 'LD2'])
lda_df['NeuronType'] = data['NeuronType']

# Calculate scaling factor for 95% confidence ellipse (Chi-square value)
scaling_factor = np.sqrt(5.991)

# Retrieve the explained variance ratio from the LDA model
explained_variance_ratio = lda.explained_variance_ratio_ * 100  # Convert to percentage

# Plotting
plt.figure(figsize=(10, 7))

# Define a specific color palette from seaborn for clarity
palette = sns.color_palette("Set1", n_colors=len(lda_df['NeuronType'].unique()))

# Map NeuronType to the palette
color_map = dict(zip(lda_df['NeuronType'].unique(), palette))

# Plot the LDA data with corrected NeuronType colors
sns.scatterplot(x='LD1', y='LD2', hue='NeuronType', data=lda_df, palette=color_map, s=100, alpha=0.7)

# Add ellipses with corrected colors
for neuron_type in lda_df['NeuronType'].unique():
    sub_data = lda_df[lda_df['NeuronType'] == neuron_type]
    covariance = np.cov(sub_data[['LD1', 'LD2']].T)
    lambda_, v = np.linalg.eig(covariance)
    lambda_ = np.sqrt(lambda_) * scaling_factor  # Adjust the eigenvalues
    ell = Ellipse(xy=(np.mean(sub_data['LD1']), np.mean(sub_data['LD2'])),
                  width=lambda_[0]*2, height=lambda_[1]*2,
                  angle=np.rad2deg(np.arccos(v[0, 0])), edgecolor=color_map[neuron_type], fc='None', lw=2, linestyle='--')
    plt.gca().add_patch(ell)

# Adjusting vector scaling for better fit
scale_factor = 2  # Same scale factor for consistency
for i, feature in enumerate(feature_names):
    plt.arrow(0, 0, lda.scalings_[i, 0]*scale_factor, lda.scalings_[i, 1]*scale_factor, color='black', head_width=0.05, head_length=0.1)
    plt.text(lda.scalings_[i, 0]*scale_factor*1.2, lda.scalings_[i, 1]*scale_factor*1.2, feature, color='black')

plt.xlabel(f'LD1 ({explained_variance_ratio[0]:.2f}%)')
plt.ylabel(f'LD2 ({explained_variance_ratio[1]:.2f}%)')
plt.title('LDA of Neuron Types')
plt.legend(title='Neuron Type')
plt.grid(True)
plt.show()

**Random Forest**

**K-means**

**Features Selection**

**Time Series**